In [12]:
!pip install -q transformers accelerate sentence-transformers llama-cpp-python bitsandbytes psutil

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 23.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.3 MB/s eta 0:00:00


In [2]:
import time
import csv
import os
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from llama_cpp import Llama
from sentence_transformers import SentenceTransformer, util

# ================= CONFIG =================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE_MODEL_PATH = "/content/models/base-model"
MERGED_MODEL_PATH = "/content/models/model-fp16"
GGUF_MODEL_PATH = "/content/quantized/model.gguf"

RESULTS_PATH = "/content/benchmarks/results.csv"
MAX_NEW_TOKENS = 128

# -------- Multi-prompt test (QA + Reasoning + Extraction) --------

EVAL_PROMPTS = {
    "qa": "What are the treatments for Heart Attack?",
    "reasoning": "Explain step by step why hypertension increases stroke risk.",
    "extraction": (
        "Extract the drug name and adverse events:\n"
        "I was prescribed Lipitor and experienced muscle pain and fatigue."
    )
}

GROUND_TRUTH = [
    "Heart attack treatment includes restoring blood flow using medications or angioplasty, followed by lifestyle changes.",
    "Hypertension damages blood vessels, increasing clot formation and increasing stroke risk.",
    "Drug: Lipitor, Adverse Events: muscle pain, fatigue."
]

print("Configuration loaded successfully")

# -------- Accuracy model --------

embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")

def semantic_accuracy(preds, refs):
    p_emb = embedder.encode(preds, convert_to_tensor=True)
    r_emb = embedder.encode(refs, convert_to_tensor=True)
    sims = util.cos_sim(p_emb, r_emb)
    return round(sims.diag().mean().item(), 3)

def get_vram_mb():
    if torch.cuda.is_available():
        return round(torch.cuda.max_memory_allocated() / 1024**2, 2)
    return 0

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Configuration loaded successfully


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
def benchmark_base_model():
    print("\n--- Benchmarking Base Model (FP16, GPU) ---")

    tokenizer = AutoTokenizer.from_pretrained(
        "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    )

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_PATH,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    prompts = list(EVAL_PROMPTS.values())
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)

    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()

    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS
        )
    end_time = time.time()

    responses = [
        tokenizer.decode(o, skip_special_tokens=True)
        for o in outputs
    ]

    total_tokens = sum(len(r.split()) for r in responses)
    total_time = end_time - start_time

    tokens_per_sec = total_tokens / total_time
    accuracy = semantic_accuracy(responses, GROUND_TRUTH)
    vram = get_vram_mb()

    del model
    torch.cuda.empty_cache()

    result = {
        "model": "Base-FP16",
        "engine": "transformers",
        "device": DEVICE.upper(),
        "batch_size": len(prompts),
        "tokens_per_sec": round(tokens_per_sec, 2),
        "latency_sec": round(total_time, 2),
        "vram_mb": vram,
        "accuracy": accuracy
    }

    print(result)
    return result


base_result = benchmark_base_model()


--- Benchmarking Base Model (FP16, GPU) ---
{'model': 'Base-FP16', 'engine': 'transformers', 'device': 'CUDA', 'batch_size': 3, 'tokens_per_sec': 32.52, 'latency_sec': 8.27, 'vram_mb': 2601.86, 'accuracy': 0.744}


In [6]:
def benchmark_merged_model():
    print("\n--- Benchmarking Fine-Tuned (Merged) Model (FP16, GPU) ---")

    tokenizer = AutoTokenizer.from_pretrained(
        "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    )

    model = AutoModelForCausalLM.from_pretrained(
        MERGED_MODEL_PATH,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    prompts = list(EVAL_PROMPTS.values())
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)

    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()

    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS
        )
    end_time = time.time()

    responses = [
        tokenizer.decode(o, skip_special_tokens=True)
        for o in outputs
    ]

    total_tokens = sum(len(r.split()) for r in responses)
    total_time = end_time - start_time

    tokens_per_sec = total_tokens / total_time
    accuracy = semantic_accuracy(responses, GROUND_TRUTH)
    vram = get_vram_mb()

    del model
    torch.cuda.empty_cache()

    result = {
        "model": "Fine-Tuned",
        "engine": "transformers",
        "device": DEVICE.upper(),
        "batch_size": len(prompts),
        "tokens_per_sec": round(tokens_per_sec, 2),
        "latency_sec": round(total_time, 2),
        "vram_mb": vram,
        "accuracy": accuracy
    }

    print(result)
    return result


merged_result = benchmark_merged_model()


--- Benchmarking Fine-Tuned (Merged) Model (FP16, GPU) ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


{'model': 'Fine-Tuned', 'engine': 'transformers', 'device': 'CUDA', 'batch_size': 3, 'tokens_per_sec': 72.68, 'latency_sec': 4.15, 'vram_mb': 2602.86, 'accuracy': 0.853}


In [7]:
def benchmark_gguf_model():
    print("\n--- Benchmarking Quantised Model (GGUF, llama.cpp, CPU) ---")

    llm = Llama(
        model_path=GGUF_MODEL_PATH,
        n_ctx=2048,
        n_threads=os.cpu_count(),
        verbose=False
    )

    start_time = time.time()
    outputs = []

    for prompt in EVAL_PROMPTS.values():
        result = llm(
            prompt,
            max_tokens=MAX_NEW_TOKENS,
            temperature=0.0
        )
        outputs.append(result["choices"][0]["text"])

    end_time = time.time()

    total_tokens = sum(len(o.split()) for o in outputs)
    total_time = end_time - start_time

    tokens_per_sec = total_tokens / total_time
    accuracy = semantic_accuracy(outputs, GROUND_TRUTH)

    result = {
        "model": "GGUF-Q8",
        "engine": "llama.cpp",
        "device": "CPU",
        "batch_size": len(EVAL_PROMPTS),
        "tokens_per_sec": round(tokens_per_sec, 2),
        "latency_sec": round(total_time, 2),
        "vram_mb": 0,
        "accuracy": accuracy
    }

    print(result)
    return result


gguf_result = benchmark_gguf_model()


--- Benchmarking Quantised Model (GGUF, llama.cpp, CPU) ---
{'model': 'GGUF-Q8', 'engine': 'llama.cpp', 'device': 'CPU', 'batch_size': 3, 'tokens_per_sec': 4.62, 'latency_sec': 49.99, 'vram_mb': 0, 'accuracy': 0.82}


In [8]:
import csv
import os

RESULTS_PATH = "/content/benchmarks/results.csv"

results = [
    base_result,
    merged_result,
    gguf_result
]

os.makedirs("/content/benchmarks", exist_ok=True)

with open(RESULTS_PATH, "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=results[0].keys()
    )
    writer.writeheader()
    writer.writerows(results)

print(f"\nBenchmark results saved successfully at:\n{RESULTS_PATH}")

# Optional: display file contents
with open(RESULTS_PATH, "r") as f:
    print("\n===== results.csv =====")
    print(f.read())


Benchmark results saved successfully at:
/content/benchmarks/results.csv

===== results.csv =====
model,engine,device,batch_size,tokens_per_sec,latency_sec,vram_mb,accuracy
Base-FP16,transformers,CUDA,3,32.52,8.27,2601.86,0.744
Fine-Tuned,transformers,CUDA,3,72.68,4.15,2602.86,0.853
GGUF-Q8,llama.cpp,CPU,3,4.62,49.99,0,0.82

